# Notebook 03 — RDO Training

## Purpose
Runs Algorithm 1 (Refusal Direction Optimization) to produce a single refusal direction `r` via gradient-based optimization. This direction is geometrically better than DIM: it satisfies both the ablation and addition properties by construction, with an explicit retain constraint to limit side effects.

## Algorithm 1 recap
```
Initialize r randomly
while not converged:
    Sample batch B ~ D
    L_ablation = CE( f_ablate(r)(p_harm),  t_answer  )   # make harmful prompts get answered
    L_addition = CE( f_add(αr̂,l)(p_safe),  t_refusal )   # make harmless prompts get refused
    L_retain   = KL( f_ablate(r)(p_safe) || f(p_safe) )  # preserve clean behavior on harmless prompts
    L = λ_abl·L_ablation + λ_add·L_addition + λ_ret·L_retain
    r ← r − η·∇L      # gradient step
    r ← r / ||r||₂    # project back to unit sphere
```

**Key implementation notes:**
- The model is **fully frozen** — only `r` (a single d-dimensional vector) is updated
- Gradient step uses Riemannian gradient descent: tangent projection before the step, then re-normalization after
- Optimizer: AdamW with `amsgrad=True` (per `rdo.py` line 667)
- Gradient accumulation: effective batch size = 16 via 16 micro-steps

**Prerequisites:** Notebooks 01 and 02 must have been run.

In [ ]:
import json
import os
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from nnsight import LanguageModel
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

torch.manual_seed(42)

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_PATH          = 'meta-llama/Llama-3.1-8B-Instruct'
DIM_DIR_PATH        = 'dim_outputs'
SPLITS_DIR          = 'data/saladbench_splits'
OUTPUT_DIR          = 'rdo_outputs'

# Training hyperparameters (from rdo.py DEFAULT_CONFIG)
LR                  = 1e-2
BATCH_SIZE          = 1             # micro-batch size
EFFECTIVE_BATCH     = 16            # effective batch (gradient accumulation)
PATIENCE            = 5             # steps without improvement before LR reduce
N_LR_REDUCE         = 2             # max LR reductions before stopping
ABLATION_LAMBDA     = 1.0
ADDITION_LAMBDA     = 0.2           # paper default; ablation loss is typically larger
RETAIN_LAMBDA       = 1.0
NUM_TARGET_TOKENS   = 30
FILTER_DATA         = True
FILTER_BATCH_SIZE   = 32

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Configuration ready.')
print(f'  Effective batch size: {EFFECTIVE_BATCH} (via {EFFECTIVE_BATCH // BATCH_SIZE}x accumulation)')
print(f'  Loss weights: λ_abl={ABLATION_LAMBDA}, λ_add={ADDITION_LAMBDA}, λ_ret={RETAIN_LAMBDA}')

## 1. Load Model and DIM Direction

In [ ]:
model = LanguageModel(MODEL_PATH, device_map='auto', torch_dtype=torch.float16)
model.requires_grad_(False)

# Load DIM artifacts
best_refusal_direction = torch.load(f'{DIM_DIR_PATH}/direction.pt').to(model.dtype)
metadata               = json.load(open(f'{DIM_DIR_PATH}/direction_metadata.json'))
best_layer             = metadata['layer']
alpha                  = best_refusal_direction.norm().detach().clone()
d_model                = model.config.hidden_size

print(f'Model hidden size : {d_model}')
print(f'DIM best layer    : {best_layer}')
print(f'Alpha (‖v_DIM‖)   : {alpha:.4f}')

## 2. Model-Specific Refusal Tokens

These are used for the bypass score metric — a fast proxy for ASR that checks the logit of the first generated token. A high logit for a non-refusal token means the model was jailbroken.

In [ ]:
# For Llama-3, 'I' is token 40 — the first token of 'I cannot...'
refusal_tokens = [40]  # Confirm with: model.tokenizer.encode('I', add_special_tokens=False)

confirm = model.tokenizer.decode(refusal_tokens)
print(f'Refusal token IDs: {refusal_tokens} → decoded: {repr(confirm)}')

## 3. Load Targets from Notebook 02

In [ ]:
LLAMA3_CHAT_TEMPLATE = (
    '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
    '{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
)
def apply_chat_template(instructions):
    return [LLAMA3_CHAT_TEMPLATE.format(instruction=inst) for inst in instructions]

harmful_train  = json.load(open(f'{SPLITS_DIR}/harmful_train.json'))
harmless_train = json.load(open(f'{SPLITS_DIR}/harmless_train.json'))
harmful_targets  = json.load(open(f'{SPLITS_DIR}/targets/harmful_targets.json'))
harmless_targets = json.load(open(f'{SPLITS_DIR}/targets/harmless_targets.json'))

harmful_train_instructions  = apply_chat_template([d['instruction'] for d in harmful_train])
harmless_train_instructions = apply_chat_template([d['instruction'] for d in harmless_train])

ablation_targets = [t['ablation'] for t in harmful_targets]
addition_targets = [t['addition'] for t in harmless_targets]
retain_targets   = [t['retain']   for t in harmless_targets]

print(f'Loaded {len(harmful_train_instructions):,} training pairs.')

## 4. Data Filtering via Bypass Score

Before training, filter out:
- **Harmful prompts** the model *already* complies with (`score ≤ 0`) — these provide no learning signal for the ablation loss
- **Harmless prompts** the model *already* refuses (`score ≥ 0`) — these contaminate the retain loss

Bypass score = `log P(non-refusal token) − log P(refusal token)` at position 0 of the completion.

In [ ]:
def get_bypass_scores(model, prompts, refusal_token_ids, batch_size=32):
    """Compute bypass score for each prompt. Positive = model currently refuses."""
    scores = []
    for i in tqdm(range(0, len(prompts), batch_size), desc='Computing bypass scores'):
        batch = prompts[i:i+batch_size]
        with torch.no_grad():
            with model.trace(batch) as tracer:
                last_logits = model.lm_head.output[:, -1].save()   # logits at last (assistant turn) position
        last_logits = last_logits.value.float()
        refusal_logit = last_logits[:, refusal_token_ids].max(dim=-1).values
        other_logit   = last_logits.mean(dim=-1)                   # mean of non-refusal logits as proxy
        scores.extend((refusal_logit - other_logit).cpu().tolist())
    return scores


if FILTER_DATA:
    print('Filtering training data...')
    harmful_scores  = get_bypass_scores(model, harmful_train_instructions,  refusal_tokens, FILTER_BATCH_SIZE)
    harmless_scores = get_bypass_scores(model, harmless_train_instructions, refusal_tokens, FILTER_BATCH_SIZE)

    # Keep harmful prompts the model still refuses (score > 0)
    # Keep harmless prompts the model answers normally (score < 0)
    harm_keep    = [i for i, s in enumerate(harmful_scores)  if s > 0]
    harmless_keep = [i for i, s in enumerate(harmless_scores) if s < 0]

    harmful_train_instructions  = [harmful_train_instructions[i]  for i in harm_keep]
    harmless_train_instructions = [harmless_train_instructions[i] for i in harmless_keep]
    ablation_targets = [ablation_targets[i] for i in harm_keep]
    addition_targets = [addition_targets[i] for i in harmless_keep]
    retain_targets   = [retain_targets[i]   for i in harmless_keep]

    # Re-balance
    n = min(len(harmful_train_instructions), len(harmless_train_instructions))
    harmful_train_instructions  = harmful_train_instructions[:n]
    harmless_train_instructions = harmless_train_instructions[:n]
    ablation_targets = ablation_targets[:n]
    addition_targets = addition_targets[:n]
    retain_targets   = retain_targets[:n]

    print(f'After filtering: {n:,} training pairs remain.')
    
    # Visualize score distributions
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, scores, title, thresh, keep_side, color in zip(
            axes,
            [harmful_scores, harmless_scores],
            ['Harmful prompts — bypass score', 'Harmless prompts — bypass score'],
            [0, 0],
            ['right', 'left'],
            ['tomato', 'steelblue']):
        ax.hist(scores, bins=60, color=color, alpha=0.75, edgecolor='white')
        ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Filter threshold')
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Bypass score (positive = model refuses)')
        ax.set_ylabel('Count')
        ax.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/bypass_score_distribution.png', dpi=150)
    plt.show()

## 5. Build Dataset and DataLoader

Each item carries the full prompt+target string (for nnsight to tokenize internally) and the label tensor (with prompt positions masked to -100 so CE loss only covers target tokens).

In [ ]:
def build_labels(tokenizer, prompt, target):
    """Tokenize prompt+target and mask prompt tokens in labels."""
    full_text    = prompt + target
    full_tokens  = tokenizer.encode(full_text,   add_special_tokens=True)
    prompt_len   = len(tokenizer.encode(prompt,  add_special_tokens=True)) - 1  # -1 for shift
    
    label = torch.tensor(full_tokens[1:], dtype=torch.long)   # shift by 1 (causal LM)
    label[:prompt_len] = -100                                  # mask prompt tokens from loss
    return full_text, label


class RDODataset(Dataset):
    def __init__(self, tokenizer, harmful_instrs, harmless_instrs,
                 ablation_tgts, addition_tgts, retain_tgts):
        self.harmful_prompts  = harmful_instrs
        self.harmless_prompts = harmless_instrs
        self.ablation_prompts, self.ablation_labels = [], []
        self.addition_prompts, self.addition_labels = [], []
        self.retain_prompts   = []

        for harm_p, safe_p, abl_t, add_t, ret_t in zip(
                harmful_instrs, harmless_instrs,
                ablation_tgts, addition_tgts, retain_tgts):
            abl_text, abl_label = build_labels(tokenizer, harm_p, abl_t)
            add_text, add_label = build_labels(tokenizer, safe_p, add_t)
            self.ablation_prompts.append(abl_text)
            self.ablation_labels.append(abl_label)
            self.addition_prompts.append(add_text)
            self.addition_labels.append(add_label)
            self.retain_prompts.append(safe_p + ret_t)

    def __len__(self):
        return len(self.harmful_prompts)

    def __getitem__(self, idx):
        return {
            'harmful_prompt':   self.harmful_prompts[idx],
            'harmless_prompt':  self.harmless_prompts[idx],
            'ablation_prompt':  self.ablation_prompts[idx],
            'ablation_labels':  self.ablation_labels[idx],
            'addition_prompt':  self.addition_prompts[idx],
            'addition_labels':  self.addition_labels[idx],
            'retain_prompt':    self.retain_prompts[idx],
        }


def custom_collate(batch):
    """Keep prompts as string lists; stack label tensors."""
    return {
        'harmful_prompt':  [b['harmful_prompt']  for b in batch],
        'harmless_prompt': [b['harmless_prompt'] for b in batch],
        'ablation_prompt': [b['ablation_prompt'] for b in batch],
        'ablation_labels': torch.stack([b['ablation_labels'] for b in batch]),
        'addition_prompt': [b['addition_prompt'] for b in batch],
        'addition_labels': torch.stack([b['addition_labels'] for b in batch]),
        'retain_prompt':   [b['retain_prompt']   for b in batch],
    }


train_dataset = RDODataset(
    model.tokenizer,
    harmful_train_instructions, harmless_train_instructions,
    ablation_targets, addition_targets, retain_targets
)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    drop_last=True, collate_fn=custom_collate
)

print(f'Dataset size: {len(train_dataset):,}')
print(f'Steps per epoch: {len(train_loader):,}')
print(f'Grad accumulation: {EFFECTIVE_BATCH // BATCH_SIZE}x → effective batch = {EFFECTIVE_BATCH}')

## 6. Loss Functions

In [ ]:
def compute_ce_loss(logits, labels):
    """
    Cross-entropy loss over the shifted sequence.
    logits: (batch, seq-1, vocab)  — model output [:, :-1]
    labels: (batch, seq-1)         — with -100 for masked prompt positions
    """
    # Pad labels to match logits shape (handles variable-length sequences)
    flat_logits = logits.reshape(-1, logits.size(-1))
    flat_labels = labels.reshape(-1)
    padding = torch.full((flat_logits.size(0),), -100,
                         device=flat_labels.device, dtype=flat_labels.dtype)
    padding[-flat_labels.size(0):] = flat_labels
    return F.cross_entropy(flat_logits, padding, ignore_index=-100)


def compute_kl_loss(logits_ablated, logits_clean, n_tokens=30):
    """
    KL( p_ablated || p_clean ) over the last n_tokens positions.
    Penalizes the ablated model for diverging from clean behavior on harmless prompts.
    Computed in float64 for numerical stability.
    """
    a = logits_ablated[:, -n_tokens:].to(torch.float64)
    b = logits_clean[:, -n_tokens:].to(torch.float64)
    return F.kl_div(
        F.log_softmax(a, dim=-1),
        F.softmax(b, dim=-1),
        reduction='batchmean'
    )


def project_onto_sphere_tangent(grad, vector):
    """Project gradient onto the tangent space of the unit sphere at `vector`.
    Removes the radial component: g_tangent = g - (g · v̂) v̂
    This is required for Riemannian gradient descent on the hypersphere.
    """
    v_hat = vector / vector.norm()
    return grad - (grad @ v_hat) * v_hat

print('Loss functions defined.')

## 7. Initialize the Refusal Direction `r`

Stored in **float32** regardless of model dtype — this is critical for gradient precision. The model's activations are cast to float32 only during the projection operation.

In [ ]:
r = torch.nn.Parameter(
    torch.randn(d_model, dtype=torch.float32, device='cuda'),
    requires_grad=True
)
with torch.no_grad():
    r.data /= r.data.norm()   # Start on the unit sphere

optimizer = torch.optim.AdamW(
    [r],
    lr=LR,
    betas=(0.9, 0.98),
    weight_decay=0.0,
    amsgrad=True            # rdo.py line 667
)

# Verify r is not the DIM direction (should be small cosine sim)
dim_hat = (best_refusal_direction.float() / best_refusal_direction.norm().float()).cpu()
print(f'r initialized. Shape: {r.shape}, norm: {r.data.norm():.4f}')
print(f'Cosine similarity with DIM direction: {(r.data.cpu() @ dim_hat).item():.4f}  (expected ~0 for random init)')

## 8. Intervention Helpers

These functions define the two intervention types used inside nnsight trace contexts.

In [ ]:
def ablate_with_r(layer_activation, r_param):
    """
    Directional ablation: x ← x − (x · r̂) r̂
    Removes the component of the activation along r.
    Applied at EVERY layer — this is what f_ablate(r) means.
    """
    r_hat = r_param / r_param.norm()
    r_hat = r_hat.to(layer_activation.dtype)
    # layer_activation shape: (batch, seq, d_model)
    proj_scalar = torch.einsum('bsd,d->bs', layer_activation, r_hat)   # (batch, seq)
    proj_vector = torch.einsum('bs,d->bsd', proj_scalar, r_hat)        # (batch, seq, d_model)
    return layer_activation - proj_vector


def add_r_at_layer(layer_activation, r_param, alpha_val, layer_idx, target_layer_idx):
    """
    Activation addition: x ← x + α·r̂  (only at target_layer_idx)
    Applied at a SINGLE layer — this is what f_add(αr̂, l_add) means.
    """
    if layer_idx != target_layer_idx:
        return layer_activation
    r_hat = r_param / r_param.norm()
    r_hat = r_hat.to(layer_activation.dtype)
    return layer_activation + alpha_val * r_hat

print('Intervention helpers defined.')

## 9. RDO Training Loop

The core optimization. Each micro-step:
1. Compute and backprop `L_ablation` (CE on harmful with ablation)
2. Compute and backprop `L_addition` (CE on harmless with addition)
3. Compute and backprop `L_retain` (KL on harmless with ablation vs clean)

After every `accumulation_steps` micro-steps:
- Project gradient onto sphere tangent
- Optimizer step + re-normalize `r`
- Check early stopping / LR reduction

In [ ]:
accumulation_steps = EFFECTIVE_BATCH // BATCH_SIZE

# Tracking
train_losses          = []
abl_loss_history      = []
add_loss_history      = []
ret_loss_history      = []
all_vectors           = []    # checkpoint r at each optimizer step

step_counter          = 0
patience_counter      = 0
lr_reduce_counter     = 0
lowest_training_loss  = float('inf')
stopped               = False

# Accumulated loss for logging
batch_abl_loss = 0.0
batch_add_loss = 0.0
batch_ret_loss = 0.0

print(f'Starting RDO training. Optimizer steps trigger every {accumulation_steps} micro-steps.')
print(f'r shape: {r.shape}  (will be projected to unit sphere after each step)\n')

for epoch in range(1):   # rdo.py DEFAULT_CONFIG epochs=1
    print(f'Epoch {epoch}')
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}'):
        ablation_prompt = batch['ablation_prompt']
        ablation_labels = batch['ablation_labels'].cuda()
        addition_prompt = batch['addition_prompt']
        addition_labels = batch['addition_labels'].cuda()
        retain_prompt   = batch['retain_prompt']
        harmful_prompt  = batch['harmful_prompt']

        # ── Loss 1: Ablation loss ─────────────────────────────────────────────
        # f_ablate(r)(p_harm) should produce t_answer (cooperative harmful response)
        with model.trace() as tracer:
            with tracer.invoke(ablation_prompt):
                for layer in model.model.layers:
                    layer.input[:] = ablate_with_r(layer.input, r)
                logits = model.lm_head.output[:, :-1]
                L_abl  = compute_ce_loss(logits, ablation_labels)
                L_abl_val = L_abl.detach().item().save()
            (ABLATION_LAMBDA * L_abl / accumulation_steps).backward()
        batch_abl_loss += L_abl_val.value

        # ── Loss 2: Addition loss ─────────────────────────────────────────────
        # f_add(αr̂, best_layer)(p_safe) should produce t_refusal (refusal on harmless input)
        with model.trace() as tracer:
            with tracer.invoke(addition_prompt):
                model.model.layers[best_layer].input[:] = (
                    model.model.layers[best_layer].input + alpha * (r / r.norm()).to(model.dtype)
                )
                logits = model.lm_head.output[:, :-1]
                L_add  = compute_ce_loss(logits, addition_labels)
                L_add_val = L_add.detach().item().save()
            (ADDITION_LAMBDA * L_add / accumulation_steps).backward()
        batch_add_loss += L_add_val.value

        # ── Loss 3: Retain loss ───────────────────────────────────────────────
        # KL( f_ablate(r)(p_safe) || f(p_safe) ) should be small
        with model.trace() as tracer:
            # Clean forward pass (no intervention)
            with tracer.invoke(retain_prompt):
                clean_logits = model.lm_head.output.detach().save()
            # Ablated forward pass
            with tracer.invoke(retain_prompt):
                for layer in model.model.layers:
                    layer.input[:] = ablate_with_r(layer.input, r)
                ablated_logits = model.lm_head.output
                L_ret = compute_kl_loss(ablated_logits, clean_logits.value, n_tokens=NUM_TARGET_TOKENS)
                L_ret_val = L_ret.detach().item().save()
            (RETAIN_LAMBDA * L_ret / accumulation_steps).backward()
        batch_ret_loss += L_ret_val.value

        step_counter += 1

        # ── Optimizer step (every accumulation_steps micro-steps) ─────────────
        if step_counter % accumulation_steps == 0:
            with torch.no_grad():
                # Riemannian gradient descent: project gradient onto tangent space
                r.grad = project_onto_sphere_tangent(r.grad, r.data)
                torch.nn.utils.clip_grad_norm_([r], max_norm=10.0)

            optimizer.step()
            optimizer.zero_grad()

            # Re-project r onto the unit sphere
            with torch.no_grad():
                r.data /= r.data.norm()

            # Record
            batch_abl_loss /= accumulation_steps
            batch_add_loss /= accumulation_steps
            batch_ret_loss /= accumulation_steps
            total_loss = batch_abl_loss + batch_add_loss + batch_ret_loss

            train_losses.append(total_loss)
            abl_loss_history.append(batch_abl_loss)
            add_loss_history.append(batch_add_loss)
            ret_loss_history.append(batch_ret_loss)
            all_vectors.append(r.detach().cpu().clone())

            opt_step = step_counter // accumulation_steps
            if opt_step % 20 == 0:
                print(f'  Step {opt_step:4d} | total={total_loss:.4f}  '
                      f'abl={batch_abl_loss:.4f}  add={batch_add_loss:.4f}  ret={batch_ret_loss:.4f}')

            # Early stopping with LR reduction
            if total_loss >= lowest_training_loss:
                patience_counter += 1
            else:
                lowest_training_loss = total_loss
                patience_counter = 0

            if patience_counter >= PATIENCE:
                if lr_reduce_counter >= N_LR_REDUCE:
                    print(f'\nEarly stop at optimizer step {opt_step}.')
                    stopped = True
                    break
                lr_reduce_counter += 1
                new_lr = optimizer.param_groups[0]['lr'] / 10
                optimizer.param_groups[0]['lr'] = new_lr
                print(f'  LR reduced to {new_lr:.2e} (reduction {lr_reduce_counter}/{N_LR_REDUCE})')
                patience_counter = 0

            batch_abl_loss = 0.0
            batch_add_loss = 0.0
            batch_ret_loss = 0.0

    if stopped:
        break

print('\nTraining complete.')

## 10. Training Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
steps = range(1, len(train_losses) + 1)

# Total loss
axes[0,0].plot(steps, train_losses, color='black', linewidth=1.5)
axes[0,0].set_title('Total Loss (L_abl + L_add + L_ret)')
axes[0,0].set_xlabel('Optimizer step')
axes[0,0].set_ylabel('Loss')

# Individual losses
axes[0,1].plot(steps, abl_loss_history, label='L_ablation', color='tomato',    linewidth=1.5)
axes[0,1].plot(steps, add_loss_history, label='L_addition', color='steelblue', linewidth=1.5)
axes[0,1].plot(steps, ret_loss_history, label='L_retain',   color='seagreen',  linewidth=1.5)
axes[0,1].set_title('Individual Loss Components')
axes[0,1].set_xlabel('Optimizer step')
axes[0,1].set_ylabel('Loss')
axes[0,1].legend()

# Cosine similarity to DIM direction over training
dim_hat_cpu = (best_refusal_direction.float().cpu() / best_refusal_direction.norm().float().cpu())
cos_to_dim  = [(v @ dim_hat_cpu).item() for v in all_vectors]
axes[1,0].plot(steps, cos_to_dim, color='purple', linewidth=1.5)
axes[1,0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1,0].set_title('Cosine Similarity: r vs DIM direction')
axes[1,0].set_xlabel('Optimizer step')
axes[1,0].set_ylabel('cos(r, v_DIM)')

# r norm (should stay at 1.0 throughout)
r_norms = [v.norm().item() for v in all_vectors]
axes[1,1].plot(steps, r_norms, color='orange', linewidth=1.5)
axes[1,1].axhline(1.0, color='black', linestyle='--', linewidth=0.8, label='Unit sphere')
axes[1,1].set_title('||r|| over training (should be 1.0)')
axes[1,1].set_xlabel('Optimizer step')
axes[1,1].set_ylabel('||r||')
axes[1,1].legend()

plt.suptitle('RDO Training Diagnostics', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Select Best Vector and Save

In [ ]:
best_idx    = int(torch.argmin(torch.tensor(train_losses)).item())
best_r      = all_vectors[best_idx]   # (d_model,), float32, on CPU, unit norm

print(f'Best optimizer step : {best_idx + 1} of {len(train_losses)}')
print(f'Best total loss     : {train_losses[best_idx]:.6f}')
print(f'Best r norm         : {best_r.norm():.6f}  (should be 1.0)')
print(f'Cosine sim to DIM   : {(best_r @ dim_hat_cpu).item():.4f}')

# Save
torch.save(best_r, f'{OUTPUT_DIR}/rdo_direction.pt')
torch.save(all_vectors, f'{OUTPUT_DIR}/all_rdo_vectors.pt')

summary = {
    'best_step':        best_idx + 1,
    'best_loss':        train_losses[best_idx],
    'total_steps':      len(train_losses),
    'best_layer':       best_layer,
    'alpha':            float(alpha),
    'cos_sim_to_dim':   float((best_r @ dim_hat_cpu).item()),
    'hyperparameters': {
        'lr': LR, 'ablation_lambda': ABLATION_LAMBDA,
        'addition_lambda': ADDITION_LAMBDA, 'retain_lambda': RETAIN_LAMBDA,
        'effective_batch': EFFECTIVE_BATCH
    }
}
json.dump(summary, open(f'{OUTPUT_DIR}/rdo_metadata.json', 'w'), indent=2)
print(f'\nSaved RDO direction → {OUTPUT_DIR}/rdo_direction.pt')
print(f'Saved metadata      → {OUTPUT_DIR}/rdo_metadata.json')
print('\nNotebook 03 complete. Proceed to Notebook 04 for evaluation.')